# 🏦 Credit Risk Prediction Model

This project builds a machine learning model to predict loan default risk using borrower financial data.

It includes:
- Data preprocessing
- Logistic Regression (baseline + improved)
- Decision Tree comparison
- Threshold tuning for business decision-making

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from google.colab import files
uploaded = files.upload()

Saving credit_risk_dataset.csv to credit_risk_dataset.csv


In [4]:
df = pd.read_csv('credit_risk_dataset.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [5]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


,0
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,895
loan_intent,0
loan_grade,0
loan_amnt,0
loan_int_rate,3116
loan_status,0
loan_percent_income,0


Observed missing values in employment length and interest rate.

In [6]:
df_clean = df.copy()

df_clean['person_emp_length'].fillna(df_clean['person_emp_length'].median(), inplace=True)
df_clean['loan_int_rate'].fillna(df_clean['loan_int_rate'].median(), inplace=True)

df_clean = df_clean[df_clean['person_age'] <= 80]
df_clean['person_emp_length'] = df_clean['person_emp_length'].clip(upper=40)

df_clean = pd.get_dummies(df_clean, drop_first=True)

/tmp/ipykernel_3588/919314784.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['person_emp_length'].fillna(df_clean['person_emp_length'].median(), inplace=True)
/tmp/ipykernel_3588/919314784.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df

In [7]:
X = df_clean.drop('loan_status', axis=1)
y = df_clean['loan_status']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)

In [11]:
model_lr_bal = LogisticRegression(max_iter=2000, class_weight='balanced')
model_lr_bal.fit(X_train_scaled, y_train)

y_pred_lr_bal = model_lr_bal.predict(X_test_scaled)

In [12]:
print("Logistic Regression (Balanced)")
print("Accuracy:", accuracy_score(y_test, y_pred_lr_bal))
print(confusion_matrix(y_test, y_pred_lr_bal))
print(classification_report(y_test, y_pred_lr_bal))

Logistic Regression (Balanced)
Accuracy: 0.8066001534919417
[[4145  948]
 [ 312 1110]]
              precision    recall  f1-score   support

           0       0.93      0.81      0.87      5093
           1       0.54      0.78      0.64      1422

    accuracy                           0.81      6515
   macro avg       0.73      0.80      0.75      6515
weighted avg       0.84      0.81      0.82      6515



In [13]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Decision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree
Accuracy: 0.9031465848042978
[[5062   31]
 [ 600  822]]
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      5093
           1       0.96      0.58      0.72      1422

    accuracy                           0.90      6515
   macro avg       0.93      0.79      0.83      6515
weighted avg       0.91      0.90      0.89      6515



In [14]:
y_probs = model_lr_bal.predict_proba(X_test_scaled)[:, 1]

thresholds = [0.3, 0.5, 0.7]

for t in thresholds:
    print(f"\nThreshold = {t}")
    y_pred_custom = (y_probs >= t).astype(int)
    print(confusion_matrix(y_test, y_pred_custom))


Threshold = 0.3
[[3126 1967]
 [ 179 1243]]

Threshold = 0.5
[[4145  948]
 [ 312 1110]]

Threshold = 0.7
[[4677  416]
 [ 510  912]]


## Key Insights

- Logistic Regression with class balancing improved default detection significantly  
- Decision Tree achieved higher accuracy but missed more defaulters  
- Threshold tuning showed trade-off between risk and customer approval  
- Lower thresholds reduce financial risk but increase rejection rate  

## Conclusion

This project demonstrates how machine learning models can be used in credit risk assessment.

It highlights the importance of:
- Prioritizing recall over accuracy
- Using probability-based decision making
- Aligning models with business strategy